In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42
K = 6

In [ ]:
HERE = Path.cwd()
CANDIDATES = [
    HERE / "data_swindon_with_commute.csv",
    HERE / "cluster_analysis" / "data_swindon_with_commute.csv",
    HERE.parent / "commuting-regression" / "data_swindon_with_commute.csv",
    HERE / "commuting-regression" / "data_swindon_with_commute.csv",
]
DATA_FILE = next(p for p in CANDIDATES if p.exists())
OUT_DIR = HERE if HERE.name == "cluster" else HERE / "cluster"

swindon = pd.read_csv(DATA_FILE)

# Optional LSOA names, only if the 137-LSOA boundary file is present
geojson = DATA_FILE.parent / "swindon_lsoa_2021_all137.geojson"
if geojson.exists():
    import geopandas as gpd
    geo_names = gpd.read_file(geojson)[["LSOA21CD", "LSOA21NM"]]
    swindon = swindon.merge(geo_names, on="LSOA21CD", how="left")

print(f"Reading: {DATA_FILE}")
print(f"Swindon LSOAs: {len(swindon)}")
swindon[["LSOA21CD", "MSOA21CD"]].head()

In [ ]:
LLM_FEATURES = [
    "log_voa_rv_2023",
    "rv_per_working_age",
    "sme_density",
    "qualification_index",
    "firm_size_diversity",
    "rv_per_employee",
    "sme_qual_interaction",
    "employment_quality",
    "modern_sector_leverage",
    "asset_growth_diversity",
]

COMMUTE_FEATURES = [
    "msoa_out_commute_share",
    "msoa_same_msoa_work_share",
    "msoa_wfh_share",
    "msoa_in_commute_share",
    "msoa_local_worker_share",
]

CLUSTER_FEATURES = LLM_FEATURES + COMMUTE_FEATURES

X = swindon[CLUSTER_FEATURES].copy()
assert X.isna().sum().sum() == 0, "Missing values in clustering features"
print(f"Clustering features: {len(CLUSTER_FEATURES)}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20)
swindon["cluster"] = kmeans.fit_predict(X_scaled)

print("Silhouette (final):", round(silhouette_score(X_scaled, swindon["cluster"]), 3))
swindon["cluster"].value_counts().sort_index()

In [ ]:
CLUSTER_LABELS = {
    0: "Employment & enterprise hub",
    1: "Deprived low-enterprise residential",
    2: "High-skilled outbound residential",
    3: "High-RV compact workplace",
    4: "Mid SME / mixed residential",
    5: "Moderate-skill local-work residential",
}
CLUSTER_DESCRIPTIONS = {
    0: "Highest mean log GVA; high SME density, employment quality and inbound commute share",
    1: "Largest cluster; lowest SME density, qualifications and mean log GVA; high outbound commute",
    2: "Highest qualification and WFH shares; high outbound commute; mid-low GVA",
    3: "Small n=5 group with extreme RV per working-age; high GVA workplace cores",
    4: "Moderate SME density and mid-high mean log GVA; mixed residential\u2013enterprise profile",
    5: "Mid SME density and qualifications; lower inbound commute than hub clusters",
}

swindon["cluster_name"] = swindon["cluster"].map(CLUSTER_LABELS)
swindon["cluster_description"] = swindon["cluster"].map(CLUSTER_DESCRIPTIONS)

gva_rank = swindon.groupby("cluster")["log_total_GVA_2023"].mean().sort_values(ascending=False)
rank_map = {cl: i + 1 for i, cl in enumerate(gva_rank.index)}
swindon["gva_rank_within_clusters"] = swindon["cluster"].map(rank_map)

swindon.groupby("cluster").agg(
    n=("LSOA21CD", "count"),
    mean_log_gva=("log_total_GVA_2023", "mean"),
    label=("cluster_name", "first"),
).round(3)

In [ ]:
out_cols = ["LSOA21CD"]
if "LSOA21NM" in swindon.columns:
    out_cols.append("LSOA21NM")
out_cols += [
    "MSOA21CD",
    "cluster",
    "cluster_name",
    "cluster_description",
    "log_total_GVA_2023",
    "gva_rank_within_clusters",
]

labels = swindon[out_cols].sort_values("LSOA21CD")
OUT_PATH = OUT_DIR / "swindon_cluster_labels.csv"
labels.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"Wrote {OUT_PATH} ({len(labels)} rows)")
labels.head(10)